# 未知制約を考慮したベイズ最適化（Outcome-constrained BO）

目的関数の最大化と、**実験するまで値が分からない2つの制約**を同時に扱います。目的関数・制約それぞれをGPで学習し、BoTorchの `qLogExpectedImprovement` に制約を渡して候補を提案します。

既知の入力制約との違い、観測後の再学習、次の候補提案、通常の制約付きBOとSafe BOの違いまで確認します。獲得関数や制約インターフェースを独自実装しません。

Theory: `docs/theory/acquisition/08_constraints.md`


In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.objective import GenericMCObjective
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import ModelListGP, SingleTaskGP
from robotorchan.optim import optimize_acqf

torch.manual_seed(0)
dtype = torch.double


## 1. 制約付きBOで何を最大化するか

通常のBOでは目的関数 `f(x)` を最大化します。制約付きBOでは、同時に `c1(x) <= 0`、`c2(x) <= 0` を満たすことが求められます。

今回の制約値は**入力xだけでは事前に分からず、実験を行って初めて観測できる**と仮定します。したがって、既知の探索範囲を制限するのではなく、制約の予測分布を学習します。

## Problem setup

objectiveは最大化します。未知制約は `c(x) <= 0` をfeasibleとします。ここでは `0.2 <= x <= 0.8` が真のfeasible領域ですが、candidate生成時にはその境界を既知情報として使いません。


In [ ]:
def evaluate(X):
    objective = 1.0 - (X - 0.72).square()
    upper_constraint = X - 0.8
    lower_constraint = 0.2 - X
    return objective, upper_constraint, lower_constraint

train_X = torch.linspace(0.0, 1.0, 9, dtype=dtype).unsqueeze(-1)
objective_Y, upper_Y, lower_Y = evaluate(train_X)


### 観測済みデータの可行性を確認する

目的関数と制約を同じ入力で測定します。真の可行領域は説明用に `0.2 <= x <= 0.8` と設定していますが、候補最適化にはその境界を直接渡しません。

観測点が制約を満たすかどうかは、**2つの制約値がともに0以下**かで判断します。

In [ ]:
observed_feasible = (upper_Y <= 0) & (lower_Y <= 0)
print("Observed feasible points:", train_X[observed_feasible.squeeze(-1)].squeeze(-1))
print("Observed feasible count:", observed_feasible.sum().item())
assert observed_feasible.any()

## Model objective and constraint outputs

objectiveと2つのconstraintを別GPとして学習し、`ModelListGP`で同じposterior sampleへまとめます。


### 目的関数と制約の3出力GP

`ModelListGP` に、目的関数用GPと2つの制約用GPをまとめます。事後サンプルの最後の次元は順に `[objective, upper_constraint, lower_constraint]` です。

ここでは3つの独立したGPを使用します。制約の値が観測できても、未知の入力における制約の**予測不確実性**は残ります。

In [ ]:
model = ModelListGP(
    SingleTaskGP(train_X, objective_Y),
    SingleTaskGP(train_X, upper_Y),
    SingleTaskGP(train_X, lower_Y),
)
model.fit()
model.eval()

test_X = torch.linspace(0.0, 1.0, 101, dtype=dtype).unsqueeze(-1)
posterior = model.posterior(test_X)
assert posterior.mean.shape[-1] == 3


### 制約の予測を可視化する

制約の事後平均と標準偏差を表示します。0を横切る領域では、実際に制約を満たすかどうかに不確実性が残ります。

図中の真の制約曲線は人工関数の説明用であり、**候補を最適化するときに使う情報ではありません**。

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    mean = posterior.mean
    std = posterior.variance.clamp_min(0).sqrt()
true_obj, true_upper, true_lower = evaluate(test_X)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
for k, (ax, truth, label) in enumerate(zip(
    axes, (true_obj, true_upper, true_lower), ("Objective", "Upper constraint", "Lower constraint")
)):
    ax.plot(test_X.squeeze(-1), truth.squeeze(-1), label="True function")
    ax.plot(test_X.squeeze(-1), mean[:, k], label="GP mean")
    ax.fill_between(test_X.squeeze(-1), mean[:, k] - 2*std[:, k], mean[:, k] + 2*std[:, k], alpha=0.2, label="Mean ± 2 SD")
    if k > 0:
        ax.axhline(0, linestyle="--", color="gray", label="Feasibility threshold")
    ax.set(xlabel="x", title=label)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## Objective and outcome constraints

`GenericMCObjective`はraw model samplesのoutput 0だけをobjective utilityにします。constraint callableはraw samplesを受け、BoTorchのMC constrained acquisition conventionに従って非正値をfeasibleとします。


### `GenericMCObjective` と `constraints` の役割

`GenericMCObjective` は3出力のサンプルから**最大化する目的関数の成分だけ**を取り出します。

`constraints` の各関数は同じ3出力サンプルを受け取り、**値が0以下なら可行**と判定します。BoTorchのMC獲得関数が目的関数の改善と制約を満たす可能性を組み合わせます。

`best_f` は**実際に観測した可行点の目的関数値の最大値**です。制約違反点の目的関数が大きくても基準にしません。

In [ ]:
objective = GenericMCObjective(lambda Y, X=None: Y[..., 0])
constraints = [
    lambda Y: Y[..., 1],
    lambda Y: Y[..., 2],
]
sampler = SobolQMCNormalSampler(torch.Size([64]), seed=27)

feasible_observed = (upper_Y <= 0) & (lower_Y <= 0)
best_f = objective_Y[feasible_observed].max()
acqf = qLogExpectedImprovement(
    model=model,
    best_f=best_f,
    sampler=sampler,
    objective=objective,
    constraints=constraints,
)


### 未知制約を探索範囲の固定境界にしない

最適化の `bounds=[0,1]` は変更しません。未知制約を `x in [0.2,0.8]` という既知制約として渡すと、問題の意味が変わってしまいます。

通常の制約付きBOは**制約違反を確率的に抑える**方法であり、次の候補が必ず可行であることを保証しません。

## Candidate generation

探索boundsは `[0, 1]` のままです。`0.2 <= x <= 0.8` をoptimizer constraintとして渡していない点が重要です。feasibilityはconstraint posteriorを通じてacquisition内で確率的に扱われます。


In [ ]:
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
candidate, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
candidate_objective, candidate_upper, candidate_lower = evaluate(candidate)
print("candidate:", candidate)
print("objective:", candidate_objective)
print("constraints:", candidate_upper, candidate_lower)
assert candidate.shape == torch.Size([1, 1])
assert torch.isfinite(acq_value).all()


### 提案点の目的値と制約値を観測する

`optimize_acqf` が返すのは次に測定する入力です。実験後には目的値だけでなく**2つの制約値も同時に取得**し、可行性を確認します。

## Update once

実験後はobjectiveだけでなくconstraint outputsも観測し、すべてのsurrogate training dataへ追加します。


In [ ]:
candidate_is_feasible = bool(((candidate_upper <= 0) & (candidate_lower <= 0)).all())
print("Candidate is feasible:", candidate_is_feasible)
print("Observed best feasible objective before update:", best_f.item())

In [ ]:
updated_X = torch.cat([train_X, candidate.detach()], dim=0)
updated_objective = torch.cat([objective_Y, candidate_objective.detach()], dim=0)
updated_upper = torch.cat([upper_Y, candidate_upper.detach()], dim=0)
updated_lower = torch.cat([lower_Y, candidate_lower.detach()], dim=0)
assert updated_X.shape[0] == train_X.shape[0] + 1
assert updated_objective.shape == updated_upper.shape == updated_lower.shape


## 再学習して次の候補を提案する

新しい観測点の3出力をすべて訓練データに追加し、3つのGPを再学習します。更新後の `best_f` は**可行な観測点だけ**から再計算します。

制約を満たさなかった観測点も、制約境界を学ぶための重要なデータです。制約違反だからといってGPの学習データから除外しません。

In [ ]:
updated_model = ModelListGP(
    SingleTaskGP(updated_X, updated_objective),
    SingleTaskGP(updated_X, updated_upper),
    SingleTaskGP(updated_X, updated_lower),
)
updated_model.fit()
updated_model.eval()
updated_feasible = (updated_upper <= 0) & (updated_lower <= 0)
updated_best_f = updated_objective[updated_feasible].max()
next_acqf = qLogExpectedImprovement(
    model=updated_model,
    best_f=updated_best_f,
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=28),
    objective=objective,
    constraints=constraints,
)
next_candidate, next_acq_value = optimize_acqf(
    acq_function=next_acqf,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("Updated feasible best:", updated_best_f.item())
print("Next constrained-BO candidate:", next_candidate.item())
print("Next acquisition value:", next_acq_value.item())
assert next_candidate.shape == (1, 1)
assert updated_X.shape[0] == train_X.shape[0] + 1

## 観測前後の目的関数・制約予測を比較する

候補を1点追加してGPを再学習した前後の事後平均を比較します。目的関数と制約は別々に予測されるため、**制約の学習結果も候補提案に影響**します。

In [ ]:
with torch.no_grad():
    updated_mean = updated_model.posterior(test_X).mean
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
for k, (ax, label) in enumerate(zip(axes, ("Objective", "Upper constraint", "Lower constraint"))):
    ax.plot(test_X.squeeze(-1), mean[:, k], label="Before observation")
    ax.plot(test_X.squeeze(-1), updated_mean[:, k], label="After observation")
    ax.axvline(candidate.item(), linestyle="--", label="New observation")
    if k > 0:
        ax.axhline(0, color="gray", linestyle=":", label="Feasibility threshold")
    ax.set(xlabel="x", title=label)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## Responsibility boundary

- **Outcome / black-box constraint**: 評価するまで未知。surrogate + constrained acquisitionで扱います。
- **Candidate / input-space constraint**: Xから既知。`CandidateConstraints`やBoTorch optimizer constraintsで探索領域を制限します。
- 両方を同時に使うこともできますが、一方を他方へ変換しません。
- constrained BOはsafe BOではありません。通常のconstrained acquisitionは制約違反を絶対に防ぐ保証を与えません。
- feasible observationが1点もない場合は、単純なfeasible incumbentベースのEIとは別の初期探索設計が必要です。


## まとめ：既知制約と未知制約の使い分け

| 目的 | 既存API | 役割 |
|---|---|---|
| 目的値・制約値のGP学習 | `SingleTaskGP` + `ModelListGP.fit()` | 3出力の予測分布 |
| 最大化する評価指標 | `GenericMCObjective` | 目的関数の出力を選択 |
| 未知制約の判定 | `qLogExpectedImprovement(constraints=...)` | 事後サンプルの制約を確率的に考慮 |
| 候補の最適化 | `robotorchan.optim.optimize_acqf` | 次の実験条件を選択 |
| 既知の入力制約 | `CandidateConstraints` | Xの可行領域を直接指定 |

**注意：** 通常のOutcome-constrained BOは未知制約の違反を完全には防ぎません。実験上の安全保証が必要な場合は、別途Safe BOの前提と方法を確認してください。